In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
methods_names = {
    "dist_linguistic_confidence": "Dist. Ling. Conf.",
    "dist_semantic_uncertainty": "Dist. Semantic Unc.",
    "dist_lnll": "Dist. Token Prob"
}

dataset_size = {
    "mmlu": 14000 ,
    "squadv2": 11900,
    "truthful_qa": 817
}

dataset_map = {
    "mmlu": "MMLU",
    "squadv2": "SQuAD2.0",
    "truthful_qa": "TruthfulQA"
}

model_name_map = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B-Inst.",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B-Inst.",
    "Qwen3-8B": "Qwen3-8B-Inst.",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B-Inst.",
    "gpt-oss-20b": "GPT-OSS-20B",
    "gpt-oss-120b": "GPT-OSS-120B",
    "gemma-4-31B-it": "Gemma-4-31B-It.",
    "Qwen3-235B-A22B-Instruct-2507-tput": "Qwen3-235B-Inst."
}

In [3]:
pct = True

# Cross domain data

In [4]:
prompt_type = "direct_qa"

In [5]:
results_dir = f"/hdd/ivny/{prompt_type}_cross_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    if model_name == "gpt-oss-120b" or model_name.lower() == "qwen3-235b-a22b-instruct-2507-tput":
        continue
    training_set, test_set = dataset_name.split("--")
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["model"] = model_name_map.get(model_name, model_name)
            record["training_set"] = dataset_map.get(training_set, training_set)
            record["test_set"] = dataset_map.get(test_set, test_set)
            record["dataset_size"] = dataset_size.get(test_set, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

In [6]:
df = pd.DataFrame(all_records).drop(columns=["dataset_size"])

In [7]:
def bootstrap_stat(group_df, cal_col, orig_col, pct=True, n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)
    cal  = group_df[cal_col].values
    orig = group_df[orig_col].values
    n = len(cal)

    mean_cal, mean_orig = cal.mean(), orig.mean()
    point = (mean_cal - mean_orig) / mean_orig if pct else mean_cal - mean_orig

    boots = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        b_cal, b_orig = cal[idx].mean(), orig[idx].mean()
        boots.append((b_cal - b_orig) / b_orig if pct else b_cal - b_orig)

    return point, float(np.std(boots))

In [8]:
fd_improvement_records, ece_improvement_records = [], []
for (train, test), grp in df.groupby(["training_set", "test_set"]):
    fd_improvement_records.append({
        "training_set": train, "test_set": test,
        "Linguistic Confidence": bootstrap_stat(grp, "calibrated_lc_rewritten_lc_faithfulness_divergence", "original_lc_faithfulness_divergence", pct=pct),
        "Token Probability":     bootstrap_stat(grp, "calibrated_tp_rewritten_lc_faithfulness_divergence", "original_lc_faithfulness_divergence", pct=pct),
        "Semantic Uncertainty":  bootstrap_stat(grp, "calibrated_su_rewritten_lc_faithfulness_divergence", "original_lc_faithfulness_divergence", pct=pct),
    })
    ece_improvement_records.append({
        "training_set": train, "test_set": test,
        "Linguistic Confidence": bootstrap_stat(grp, "calibrated_lc_rewritten_lc_generalised_ECE", "original_lc_generalised_ECE", pct=pct),
        "Token Probability":     bootstrap_stat(grp, "calibrated_tp_rewritten_lc_generalised_ECE", "original_lc_generalised_ECE", pct=pct),
        "Semantic Uncertainty":  bootstrap_stat(grp, "calibrated_su_rewritten_lc_generalised_ECE", "original_lc_generalised_ECE", pct=pct),
    })

# In domain diagonal data

In [9]:
results_dir = f"/hdd/ivny/{prompt_type}_in_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    if model_name == "gpt-oss-120b" or model_name.lower() == "qwen3-235b-a22b-instruct-2507-tput":
        continue
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["dataset"] = dataset_map.get(dataset_name, dataset_name)
            record["model"] = model_name_map.get(model_name, model_name)
            record["dataset_size"] = dataset_size.get(dataset_name, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

In [10]:
raw_in = pd.DataFrame(all_records).drop(columns=["dataset_size"])

in_domain_fd_records, in_domain_ece_records = [], []
for dataset, grp in raw_in.groupby("dataset"):
    in_domain_fd_records.append({
        "training_set": dataset, "test_set": dataset,
        "Linguistic Confidence": bootstrap_stat(grp, "calibrated_lc_rewritten_lc_faithfulness_divergence", "original_lc_faithfulness_divergence", pct=pct),
        "Token Probability":     bootstrap_stat(grp, "calibrated_tp_rewritten_lc_faithfulness_divergence", "original_lc_faithfulness_divergence", pct=pct),
        "Semantic Uncertainty":  bootstrap_stat(grp, "calibrated_su_rewritten_lc_faithfulness_divergence", "original_lc_faithfulness_divergence", pct=pct),
    })
    in_domain_ece_records.append({
        "training_set": dataset, "test_set": dataset,
        "Linguistic Confidence": bootstrap_stat(grp, "calibrated_lc_rewritten_lc_generalised_ECE", "original_lc_generalised_ECE", pct=pct),
        "Token Probability":     bootstrap_stat(grp, "calibrated_tp_rewritten_lc_generalised_ECE", "original_lc_generalised_ECE", pct=pct),
        "Semantic Uncertainty":  bootstrap_stat(grp, "calibrated_su_rewritten_lc_generalised_ECE", "original_lc_generalised_ECE", pct=pct),
    })

In [11]:
raw_in[raw_in["dataset"] == "MMLU"]

,original_lc_generalised_ECE,original_lc_faithfulness_divergence,original_lc_ece_mean,original_lc_dAUROC,original_lc_auroc_mean,original_tp_generalised_ECE,original_tp_faithfulness_divergence,original_tp_ece_mean,original_tp_dAUROC,original_tp_auroc_mean,...,calibrated_su_rewritten_lc_ece_mean,calibrated_su_rewritten_lc_dAUROC,calibrated_su_rewritten_lc_auroc_mean,calibrated_su_beta_guided_lc_generalised_ECE,calibrated_su_beta_guided_lc_faithfulness_divergence,calibrated_su_beta_guided_lc_ece_mean,calibrated_su_beta_guided_lc_dAUROC,calibrated_su_beta_guided_lc_auroc_mean,dataset,model
0,0.239395,1.198330,0.207488,0.53100,0.536319,0.381067,335.579049,0.379346,0.63730,0.658290,...,0.074639,0.5501,0.571734,0.141643,0.674156,0.135956,0.5459,0.566116,MMLU,Mistral-7B-Inst.
1,0.284387,1.047344,0.246779,0.45605,0.394317,0.118232,112.963519,0.118113,0.59690,0.615128,...,0.040530,0.5597,0.572478,0.121950,0.441685,0.087919,0.5233,0.524304,MMLU,Gemma-4-31B-It.
2,0.190048,0.925248,0.137413,0.52970,0.556111,0.260864,4.656756,0.251987,0.64495,0.673481,...,0.061831,0.6568,0.705205,0.108194,0.624603,0.089345,0.5949,0.632701,MMLU,Llama-3.1-8B-Inst.
3,0.151864,0.579284,0.094998,0.50620,0.511461,0.148043,1.875741,0.099184,0.53325,0.538044,...,0.071491,0.7192,0.748724,0.086329,0.528301,0.041048,0.6060,0.623581,MMLU,GPT-OSS-20B
4,0.166724,0.901899,0.089968,0.54870,0.565632,0.280934,223.551016,0.277312,0.52290,0.505690,...,0.113765,0.5543,0.579526,0.117603,0.737846,0.091175,0.5289,0.552656,MMLU,Qwen3-8B-Inst.


# Format latex table

In [14]:
def generate_latex_table(faithfulness_data, ece_data, pct):
    def fmt(value, pct=False):
        if value is None:
            return r"-"
        mean, std = value if isinstance(value, tuple) else (value, None)
        if pct:
            pct_val = mean * 100
            color = "green!70!black" if pct_val < 0 else "red!70!black"
            s = rf"\textcolor{{{color}}}{{$\Delta${abs(pct_val):.1f}"
            if std is not None:
                s += rf"$\pm${std * 100:.1f}"
            s += r"\%}"
            return s
        else:
            color = "green!70!black" if mean < 0 else "red!70!black"
            s = rf"\textcolor{{{color}}}{{$\Delta${abs(mean):.4f}"
            if std is not None:
                s += rf"$\pm${std:.4f}"
            s += r"}"
            return s

    def lookup(data, train, test):
        for row in data:
            if row["training_set"] == train and row["test_set"] == test:
                return row
        return None

    estimators = [
        ("Linguistic\\\\Confidence", "Linguistic Confidence"),
        ("Token\\\\Probability",     "Token Probability"),
        ("Semantic\\\\Uncertainty",  "Semantic Uncertainty"),
    ]
    datasets = ["MMLU", "SQuAD2.0", "TruthfulQA"]

    def build_block(data, metric_label):
        lines = []
        lines.append(rf"\multirow{{9}}{{=}}{{\textbf{{{metric_label}}}}}")

        for ei, (est_display, est_key) in enumerate(estimators):
            lines.append(rf"& \multirow{{3}}{{=}}{{{est_display}}}")

            for ti, train in enumerate(datasets):
                cells = []
                for col in datasets:
                    row = lookup(data, train, col)
                    cells.append(fmt(row.get(est_key), pct=pct) if row else "-")

                cell_str = " & ".join(cells)
                if ti == 0:
                    lines.append(rf"& {train} & {cell_str} \\")
                else:
                    lines.append(rf"& & {train} & {cell_str} \\")

            if ei < len(estimators) - 1:
                lines.append(r"\cmidrule(lr){2-6}")

        return lines

    caption = (
        r"In-domain and cross-domain linguistic-space calibration metric percentage changes for both "
        r"Faithfulness Divergence and generalised ECE. We report percentage change relative to the "
        r"pre-calibration metrics (mean\,$\pm$\,std across models). "
        r"Green text indicates calibration improvement (lower error), red indicates deterioration. "
        r"Semantic uncertainty is the strongest signal for RALC in improving both calibration and faithfulness across all three benchmarks in both in-domain and cross-domain settings."
    ) if pct else (
        r"In-domain and cross-domain linguistic-space calibration metric value changes for both "
        r"Faithfulness Divergence and generalised ECE. We report the value change relative to the "
        r"pre-calibration metrics (mean\,$\pm$\,std across models). "
        r"Green text indicates calibration improvement (lower error), red indicates deterioration. "
        r"Semantic uncertainty is the strongest signal for RALC in improving both calibration and faithfulness across all three benchmarks in both in-domain and cross-domain settings."
    )

    tabular_lines = [
        r"\begin{tabular}{p{2cm}p{2cm}lccc}",
        r"\toprule",
        r"\textbf{Metric} & \textbf{Signal} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\",
        r"\midrule",
    ]
    tabular_lines.extend(build_block(faithfulness_data, "Faithfulness\\\\Divergence\\\\Mean\\\\Reduction"))
    tabular_lines.append(r"\midrule")
    tabular_lines.extend(build_block(ece_data, "Generalised\\\\ECE\\\\Mean\\\\Reduction"))
    tabular_lines += [
        r"\bottomrule",
        r"\end{tabular}",
    ]

    latex_lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        r"\label{tab:cross-domain-calibration}",
        r"\resizebox{\linewidth}{!}{",
        "\n".join(tabular_lines),
        r"}",
        r"\end{table}",
    ]

    return "\n".join(latex_lines)

print(generate_latex_table(
    fd_improvement_records  + in_domain_fd_records,
    ece_improvement_records + in_domain_ece_records,
    pct=False,
))

\begin{table}[t]
\centering
\caption{In-domain and cross-domain linguistic-space calibration metric value changes for both Faithfulness Divergence and generalised ECE. We report the value change relative to the pre-calibration metrics (mean\,$\pm$\,std across models). Green text indicates calibration improvement (lower error), red indicates deterioration. Semantic uncertainty is the strongest signal for RALC in improving both calibration and faithfulness across all three benchmarks in both in-domain and cross-domain settings.}
\label{tab:cross-domain-calibration}
\resizebox{\linewidth}{!}{
\begin{tabular}{p{2cm}p{2cm}lccc}
\toprule
\textbf{Metric} & \textbf{Signal} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\
\midrule
\multirow{9}{=}{\textbf{Faithfulness\\Divergence\\Mean\\Reduction}}
& \multirow{3}{=}{Linguistic\\Confidence}
& MMLU & \textcolor{green!70!black}{$\Delta$0.1136$\pm$0.1924} & \textcolor{green!70!black}{$\Delta$0.3459$\pm$0.1157} & \textcolor{green!70!black}